# Z4 115M — Kaggle TPU smoke

This runs the measured **`kaggle10m_nomtp`** profile on a Kaggle TPU:

- 115,238,237 parameters (MTP compute skipped)
- bf16 autocast on XLA
- 600-second training cap
- WikiText-2 train+valid, 24k held-out test tokens

`TPU_MODE = "single"` uses one XLA device with `train_v3.py --device xla`.
`TPU_MODE = "multi"` uses the experimental 8-replica wrapper
`z3/train_tpu.py`.  Start with single; the multi-core wrapper is not yet
TPU-validated.

This is an early-training smoke, not a frontier claim.


In [ ]:
!nvidia-smi 2>/dev/null || true
import os, sys, subprocess
os.environ.setdefault("PJRT_DEVICE", "TPU")
print({k: os.environ.get(k) for k in ("TPU_NAME", "COLAB_TPU_ADDR", "PJRT_DEVICE")})
subprocess.run([sys.executable, "-c", "import torch; print('image torch', torch.__version__)"])

# Kaggle's TPU image ships torch 2.11, but the libtpu index only has
# torch_xla 2.8/2.9.  Install torch 2.9.0 from PyPI (the ABI the torch_xla
# wheel is built against), then torch_xla without its broken libtpu==0.0.21
# extra pin, and use the available libtpu 0.0.21.1.
steps = [
    [sys.executable, "-m", "pip", "install", "-q", "torch==2.9.0"],
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "torch_xla==2.9.0",
     "-f", "https://storage.googleapis.com/libtpu-releases/index.html"],
    [sys.executable, "-m", "pip", "install", "-q", "libtpu==0.0.21.1",
     "-f", "https://storage.googleapis.com/libtpu-releases/index.html"],
    [sys.executable, "-m", "pip", "install", "-q", "absl-py", "numpy", "pyyaml", "requests", "tpu-info"],
]
for command in steps:
    print("$", " ".join(command), flush=True)
    subprocess.run(command, check=True)
check = subprocess.run([
    sys.executable, "-c",
    "import torch; print('torch', torch.__version__); "
    "import torch_xla; print('torch_xla', torch_xla.__version__); "
    "import torch_xla.core.xla_model as xm; "
    "print('devices', xm.get_xla_supported_devices()); "
    "print('device', xm.xla_device())",
], check=True)


In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO = Path("/kaggle/working/nanlm_experimental")
if not (REPO / "z3" / "train_v3.py").is_file():
    subprocess.run(["git","clone","--depth","1","https://github.com/shemshallah/nanlm_experimental.git",str(REPO)],check=True)
else:
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
os.chdir(REPO)
print(REPO)


In [ ]:
subprocess.run([sys.executable,"-m","pip","install","-q","tokenizers","numpy","pyarrow"],check=True)


In [ ]:
import urllib.request
Path("data").mkdir(exist_ok=True)
parts=[]
for split in ("train","valid"):
    url=f"https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/{split}.txt"
    dest=Path(f"data/wikitext_{split}.txt")
    urllib.request.urlretrieve(url,dest)
    parts.append(dest.read_text(encoding="utf-8"))
corpus=Path("data/wikitext_corpus.txt")
corpus.write_text("\n\n".join(parts),encoding="utf-8")
print(corpus, corpus.stat().st_size)
subprocess.run([
    sys.executable,"z3/build_domain_bins.py",
    "--corpus","wikitext=data/wikitext_corpus.txt",
    "--out","data/wikitext_bins",
    "--max-bytes-per-domain","12000000",
    "--train-fraction","0.98","--valid-fraction","0.01",
],check=True)


In [ ]:
TPU_MODE = "single"   # "single" or "multi"
OUT = Path("/kaggle/working/runs/z3_115m_tpu")
sources = [
    "--preset","kaggle10m_nomtp",
    "--corpus","data/wikitext_bins/train.bin",
    "--val-corpus","data/wikitext_bins/valid.bin",
    "--data-manifest","data/wikitext_bins/manifest.json",
    "--batch-size","4",
]
if TPU_MODE == "single":
    cmd=[sys.executable,"z3/train_v3.py","--device","xla","--amp",*sources,"--out",str(OUT)]
else:
    cmd=[sys.executable,"z3/train_tpu.py","--nprocs","8","--out",str(OUT),*sources]
print(" ".join(cmd))
try:
    subprocess.run(cmd,check=True)
except subprocess.CalledProcessError:
    print("TPU launch failed; retrying the single-device path")
    subprocess.run([sys.executable,"z3/train_v3.py","--device","xla","--amp",*sources,"--out",str(OUT)],check=True)


In [ ]:
subprocess.run([
    sys.executable,"z3/eval_domains.py",
    "--checkpoint",str(OUT/"best.pt"),
    "--test-dir","data/wikitext_bins/test",
    "--data-manifest","data/wikitext_bins/manifest.json",
    "--tokenizer","layered","--document-framing","--sidecar","--require-sidecar",
    "--ctx","512","--batch-size","4","--device","xla",
    "--out",str(OUT/"test_domains.json"),
],check=True)


In [ ]:
import json
result=json.loads((OUT/"result.json").read_text())
test=json.loads((OUT/"test_domains.json").read_text())
print(json.dumps({
    "params":result.get("params"),
    "steps":result.get("steps_completed"),
    "tokens_seen":result.get("tokens_seen"),
    "nonfinite_skips":result.get("nonfinite_skips"),
    "best_step":(result.get("best") or {}).get("step"),
    "best_bpb":((result.get("best") or {}).get("metrics") or {}).get("bits_per_byte"),
    "final_bpb":(result.get("final") or {}).get("bits_per_byte"),
    "macro_test_bpb":test.get("macro_bits_per_byte"),
    "feedback":(result.get("final") or {}).get("feedback"),
},indent=2,sort_keys=True))
